# 04 — Portfolio Construction

## AI-Based NIFTY 50 Portfolio Risk Prediction & Early Warning System

### Goal
Convert stock-level returns into a portfolio-level time series that we can later use to define and predict future drawdown risk.

### Primary portfolio
**Daily-rebalanced equal-weight portfolio across stocks with a valid return on each date.**

For date `t`:

`weight_i,t = 1 / N_t`

and:

`Portfolio_Return_t = sum(weight_i,t × Return_i,t)`

### Important limitation
This is **not a reconstruction of historical official NIFTY 50 index membership**. The supplied dataset contains 49 stocks with different historical start dates. That survivorship/constituent limitation is documented and will remain explicit in the final project.


## 1. Why equal weight?

Equal weight is our baseline because it is:

- simple
- transparent
- easy to reproduce
- free from an optimization model
- less likely to introduce unnecessary assumptions

Later, if useful, we can compare it with inverse-volatility or sector-constrained portfolios.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

INPUT_PATH = Path("../data/processed/stock_features.csv")
df = pd.read_csv(INPUT_PATH, parse_dates=["Date"])
df = df.sort_values(["Date", "Ticker"]).reset_index(drop=True)

print("Input shape:", df.shape)
print("Stocks:", df["Ticker"].nunique())
print("Date range:", df["Date"].min(), "to", df["Date"].max())


## 2. Select valid daily stock returns

We use `Return_1D` created in Notebook 03.

A missing return is **not** treated as zero because missing information does not mean the stock earned 0%.


In [ ]:
returns = df.loc[
    df["Return_1D"].notna(),
    ["Date", "Ticker", "Sector", "Return_1D"]
].copy()

returns["Available_Count"] = returns.groupby("Date")["Ticker"].transform("count")
returns["Weight"] = 1.0 / returns["Available_Count"]
returns["Weighted_Return"] = returns["Weight"] * returns["Return_1D"]

print("Usable stock-return rows:", len(returns))
print("Portfolio dates:", returns["Date"].nunique())


## 3. Calculate daily equal-weight portfolio returns

On each date, all stocks with valid returns receive the same weight.

This is a **daily rebalanced** portfolio: weights are reset to equal each trading day.


In [ ]:
portfolio_daily = (
    returns.groupby("Date")
    .agg(
        Portfolio_Return=("Weighted_Return", "sum"),
        Number_of_Stocks=("Ticker", "nunique"),
        Weight_Check=("Weight", "sum")
    )
    .reset_index()
)

portfolio_daily.head()


## 4. Validate weights

The daily weights should sum to 1.

A value close to 1 confirms that the daily equal-weight portfolio is correctly normalized.


In [ ]:
print(portfolio_daily["Weight_Check"].describe())
print("Maximum deviation from 1:",
      (portfolio_daily["Weight_Check"] - 1).abs().max())


## 5. Portfolio value and drawdown

We normalize the starting portfolio value to `1`.

`Portfolio_Value_t = Portfolio_Value_(t-1) × (1 + Portfolio_Return_t)`

`Current_Drawdown = Portfolio_Value / Running_Peak - 1`


In [ ]:
portfolio_daily["Portfolio_Value"] = (1 + portfolio_daily["Portfolio_Return"]).cumprod()
portfolio_daily["Running_Peak"] = portfolio_daily["Portfolio_Value"].cummax()
portfolio_daily["Current_Drawdown"] = (
    portfolio_daily["Portfolio_Value"] / portfolio_daily["Running_Peak"] - 1
)

portfolio_daily[[
    "Date", "Portfolio_Return", "Portfolio_Value", "Current_Drawdown"
]].tail()


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(portfolio_daily["Date"], portfolio_daily["Portfolio_Value"])
plt.title("Equal-Weight Portfolio Growth")
plt.xlabel("Date")
plt.ylabel("Portfolio Value (Starting at 1)")
plt.tight_layout()
plt.show()


## 6. Rolling portfolio volatility

We calculate 5-, 20-, and 60-trading-day standard deviation of portfolio returns.


In [ ]:
for window in [5, 20, 60]:
    portfolio_daily[f"Volatility_{window}D"] = (
        portfolio_daily["Portfolio_Return"]
        .rolling(window, min_periods=window)
        .std()
    )

portfolio_daily[[
    "Date", "Volatility_5D", "Volatility_20D", "Volatility_60D"
]].tail()


## 7. Portfolio momentum

We calculate recent portfolio performance over 5 and 20 trading days.


In [ ]:
portfolio_daily["Return_5D"] = portfolio_daily["Portfolio_Value"].pct_change(5)
portfolio_daily["Return_20D"] = portfolio_daily["Portfolio_Value"].pct_change(20)


## 8. Trailing drawdown

We calculate drawdown relative to the maximum portfolio value observed in trailing 20- and 60-day windows.

These are historical/current features only.


In [ ]:
for window in [20, 60]:
    rolling_peak = portfolio_daily["Portfolio_Value"].rolling(window, min_periods=1).max()
    portfolio_daily[f"Max_Drawdown_{window}D"] = (
        portfolio_daily["Portfolio_Value"] / rolling_peak - 1
    )


## 9. Stock concentration

`HHI = sum(weight²)`

Higher HHI means greater concentration.

We also track largest-stock and top-5-stock weights.


In [ ]:
concentration = (
    returns.groupby("Date")
    .apply(
        lambda g: pd.Series({
            "Stock_HHI": (g["Weight"] ** 2).sum(),
            "Largest_Stock_Weight": g["Weight"].max(),
            "Top_5_Stock_Weight": g.nlargest(
                min(5, len(g)), "Weight"
            )["Weight"].sum()
        }),
        include_groups=False
    )
    .reset_index()
)

portfolio_daily = portfolio_daily.merge(concentration, on="Date", how="left")


## 10. Sector exposure and concentration

We aggregate stock weights by sector and calculate:

- largest sector weight
- sector HHI


In [ ]:
sector_weights = (
    returns.groupby(["Date", "Sector"])["Weight"]
    .sum()
    .reset_index(name="Sector_Weight")
)

sector_concentration = (
    sector_weights.groupby("Date")
    .agg(
        Largest_Sector_Weight=("Sector_Weight", "max"),
        Sector_HHI=("Sector_Weight", lambda x: (x ** 2).sum())
    )
    .reset_index()
)

portfolio_daily = portfolio_daily.merge(
    sector_concentration, on="Date", how="left"
)

sector_weights.head()


## 11. Stock availability through time

The source dataset does not contain the same number of stocks throughout the full period.

We record how many stocks contribute to the portfolio on each date.

This is a methodological limitation, not something to hide.


In [ ]:
coverage = (
    returns.groupby("Date")["Ticker"]
    .nunique()
    .rename("Available_Stocks")
    .reset_index()
)

print(coverage["Available_Stocks"].describe())

plt.figure(figsize=(12, 5))
plt.plot(coverage["Date"], coverage["Available_Stocks"])
plt.title("Number of Stocks Available for Portfolio Construction")
plt.xlabel("Date")
plt.ylabel("Available Stocks")
plt.tight_layout()
plt.show()


## 12. Portfolio quality checks

We verify:

- one row per date
- no missing portfolio return
- weights sum to 1
- no duplicate dates

No future information is used in portfolio-return construction.


In [ ]:
print("Duplicate portfolio dates:",
      portfolio_daily["Date"].duplicated().sum())
print("Missing portfolio returns:",
      portfolio_daily["Portfolio_Return"].isna().sum())
print("Maximum weight error:",
      (portfolio_daily["Weight_Check"] - 1).abs().max())


## 13. Portfolio summary statistics


In [ ]:
summary_stats = portfolio_daily[[
    "Portfolio_Return", "Volatility_5D", "Volatility_20D", "Volatility_60D",
    "Return_5D", "Return_20D", "Current_Drawdown",
    "Max_Drawdown_20D", "Max_Drawdown_60D",
    "Stock_HHI", "Largest_Stock_Weight", "Top_5_Stock_Weight",
    "Largest_Sector_Weight", "Sector_HHI"
]].describe(percentiles=[.01,.05,.25,.50,.75,.95,.99]).T

summary_stats


## 14. Save outputs

- `data/processed/portfolio_daily_baseline.csv`
- `data/processed/portfolio_stock_weights.csv`
- `data/processed/portfolio_sector_weights.csv`
- `reports/portfolio_construction_summary.csv`


In [ ]:
portfolio_daily.to_csv("../data/processed/portfolio_daily_baseline.csv", index=False)

returns[[
    "Date","Ticker","Sector","Return_1D",
    "Available_Count","Weight","Weighted_Return"
]].to_csv("../data/processed/portfolio_stock_weights.csv", index=False)

sector_weights.to_csv("../data/processed/portfolio_sector_weights.csv", index=False)

summary = pd.DataFrame({
    "Metric": [
        "Portfolio dates","First portfolio date","Last portfolio date",
        "Minimum available stocks","Median available stocks","Maximum available stocks",
        "Dates with >=45 stocks","Dates with >=40 stocks",
        "Mean daily portfolio return","Daily portfolio volatility",
        "Final cumulative portfolio value","Maximum historical drawdown",
        "Maximum stock HHI","Maximum sector HHI"
    ],
    "Value": [
        len(portfolio_daily), portfolio_daily["Date"].min(), portfolio_daily["Date"].max(),
        coverage["Available_Stocks"].min(), coverage["Available_Stocks"].median(),
        coverage["Available_Stocks"].max(),
        int((coverage["Available_Stocks"] >= 45).sum()),
        int((coverage["Available_Stocks"] >= 40).sum()),
        portfolio_daily["Portfolio_Return"].mean(),
        portfolio_daily["Portfolio_Return"].std(),
        portfolio_daily["Portfolio_Value"].iloc[-1],
        portfolio_daily["Current_Drawdown"].min(),
        portfolio_daily["Stock_HHI"].max(),
        portfolio_daily["Sector_HHI"].max()
    ]
})

summary.to_csv("../reports/portfolio_construction_summary.csv", index=False)
summary


# Conclusion

We now have a reproducible baseline portfolio.

### Primary portfolio
**Daily-rebalanced equal-weight portfolio across stocks with valid same-day returns.**

The next notebook is:

`05_risk_target_creation.ipynb`

It will look **forward** from each date and create the labels the ML model must predict:

- 3% future drawdown
- 5% future drawdown
- 10% future drawdown

Crucial rule:

**Features use information up to date t; labels use only dates after t.**
